In [3]:
# %% [markdown]
# # 📱 Google Play Review Fetcher (Project 1 - Phase 1)
# Fetches real customer feedback across 5 popular apps, maps ratings to sentiment, and exports a clean CSV.

# %% [code]
# Install dependencies (safe to run multiple times)
# !pip install -q google-play-scraper pandas tqdm

from google_play_scraper import reviews, Sort
import pandas as pd
from tqdm import tqdm
import time
import random
import os

# 🔧 Configuration
APP_LIST = [
    {"id": "com.jumia.android", "name": "Jumia (E-commerce)"},
    {"id": "com.glovo", "name": "Glovo (Food Delivery)"},
    {"id": "com.ubercab", "name": "Uber (Ride-hailing)"},
    {"id": "com.booking", "name": "Booking.com (Travel)"},
    {"id": "com.amazon.mShop.android.shopping", "name": "Amazon (Retail)"}
]

REVIEWS_PER_APP = 500
LANG = "en"
COUNTRY = "us"
SORT = Sort.MOST_RELEVANT

# 📁 Paths: Notebook is in notebooks/, data goes to root/data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
OUTPUT_PATH = os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv")

# Create data folder if missing
os.makedirs(DATA_DIR, exist_ok=True)

# ⚡ Skip if already fetched (saves time & API calls)
if os.path.exists(OUTPUT_PATH):
    print(f"✅ Data already exists: {OUTPUT_PATH}")
    print("💡 Delete the file to re-fetch fresh reviews.")
else:
    all_reviews = []

    print("🔍 Fetching real customer reviews across 5 apps...\n")
    for app in tqdm(APP_LIST, desc="Apps", unit="app"):
        try:
            result, _ = reviews(
                app["id"], 
                lang=LANG, 
                country=COUNTRY, 
                sort=SORT, 
                count=REVIEWS_PER_APP
            )
            for r in result:
                r["app_name"] = app["name"]
                r["app_id"] = app["id"]
            all_reviews.extend(result)
            print(f"✅ {app['name']}: {len(result)} reviews fetched")
            time.sleep(random.uniform(1.5, 3.0))  # Respectful delay
        except Exception as e:
            print(f"⚠️ Failed to fetch {app['name']}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(all_reviews)
    if df.empty:
        raise RuntimeError("No reviews fetched. Check app IDs, network, or library version.")

    # 🏷️ Automatic Sentiment Labeling from Ratings
    def label_sentiment(score):
        if score >= 4.0: return "positive"
        elif score <= 2.0: return "negative"
        else: return "neutral"

    df["sentiment"] = df["score"].apply(label_sentiment)

    # 🧼 Basic Cleaning & Column Selection
    df_clean = df[["app_name", "userName", "content", "score", "sentiment", "at"]].copy()
    df_clean = df_clean.rename(columns={"content": "text", "at": "date"})
    df_clean = df_clean.dropna(subset=["text"])
    df_clean = df_clean[df_clean["text"].str.strip() != ""].reset_index(drop=True)

    # 💾 Export
    df_clean.to_csv(OUTPUT_PATH, index=False)

    print(f"\n✅ Saved {len(df_clean)} reviews to {OUTPUT_PATH}")
    
# 📊 Show stats (works whether fetched or loaded)
df_stats = pd.read_csv(OUTPUT_PATH) if not os.path.exists(OUTPUT_PATH) else pd.read_csv(OUTPUT_PATH)
print("📊 Sentiment Distribution:")
print(df_stats["sentiment"].value_counts())
print("\n📈 App-wise Distribution:")
print(df_stats["app_name"].value_counts())

✅ Data already exists: c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\notebooks\..\data\gplay_multi_app_reviews.csv
💡 Delete the file to re-fetch fresh reviews.
📊 Sentiment Distribution:
sentiment
negative    1812
positive     446
neutral      242
Name: count, dtype: int64

📈 App-wise Distribution:
app_name
Jumia (E-commerce)       500
Glovo (Food Delivery)    500
Uber (Ride-hailing)      500
Booking.com (Travel)     500
Amazon (Retail)          500
Name: count, dtype: int64


In [4]:
#!pip install pandas numpy scikit-learn spacy tqdm matplotlib
#!pip install google_play_scraper


In [5]:
import pandas as pd
import os

# Fix: Notebook is in notebooks/, so go up one level to reach data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")

# Load ONLY 5,000 from Yelp (fast, sufficient for DistilBERT)
yelp_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), 
                      header=None, 
                      names=["label", "text"], 
                      nrows=5000)

yelp_df["label"] = yelp_df["label"].map({1: "negative", 2: "positive"})
yelp_df["source"] = "yelp_public"

# Load your Google Play data (2,500 reviews)
gplay_df = pd.read_csv(os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv"))
gplay_df = gplay_df.rename(columns={"sentiment": "label", "content": "text"})
gplay_df["source"] = "google_play"

# Merge
df = pd.concat([yelp_df, gplay_df], ignore_index=True)
df = df.rename(columns={"label": "sentiment", "text": "review"})

print(f"Total: {len(df)} reviews")
print(df["sentiment"].value_counts())
print(f"Sources: {df['source'].value_counts().to_dict()}")

Total: 7500 reviews
sentiment
negative    4424
positive    2834
neutral      242
Name: count, dtype: int64
Sources: {'yelp_public': 5000, 'google_play': 2500}


In [6]:
print(df.head)

<bound method NDFrame.head of      sentiment                                             review  \
0     negative  Unfortunately, the frustration of being Dr. Go...   
1     positive  Been going to Dr. Goldberg for over 10 years. ...   
2     negative  I don't know what Dr. Goldberg was like before...   
3     negative  I'm writing this review to give you a heads up...   
4     positive  All the food is great here. But the best thing...   
...        ...                                                ...   
7495   neutral  If I pause my activity for a short time, the a...   
7496  negative  it's a waste paying Amazon prime when items ar...   
7497  negative  As of the recent update, my Amazon gift card i...   
7498  negative  having trouble trying to cancel a 2 monthly or...   
7499  positive  Amazon shopping batabase at your finger-tips, ...   

           source         app_name          userName  score  \
0     yelp_public              NaN               NaN    NaN   
1     yelp_publ

In [7]:
# Nettoyage des Textes
# Normalisation, suppression des doublons, mots vides et caractères parasites.

# %% [code]
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm

# Télécharger les ressources NLTK (exécution unique)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def normalize_text(text):
    """Lowercase + suppression URLs, mentions, hashtags, caractères non-alphabétiques"""
    if pd.isna(text): return ""
    t = str(text).lower()
    t = re.sub(r'http\S+|www\S+|@\S+|#\S+|[^a-z\s]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

print("🧹 1/3 Normalisation...")
df['clean_review'] = df['review'].apply(normalize_text)

print("🧹 2/3 Suppression des doublons...")
before = len(df)
df = df.drop_duplicates(subset=['clean_review']).reset_index(drop=True)
print(f"   → {before - len(df)} doublons retirés.")

print(" 3/3 Tokenisation + suppression des stopwords...")
tqdm.pandas()
df['tokens'] = df['clean_review'].progress_apply(
    lambda x: [w for w in word_tokenize(x) if w not in stop_words and len(w) > 2]
)
# Rejoindre les tokens pour TF-IDF
df['final_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Filtrer les textes vides ou trop courts après nettoyage
df = df[df['final_text'].str.len() > 5].reset_index(drop=True)
print(f"✅ Nettoyage terminé : {len(df)} reviews prêtes.")

🧹 1/3 Normalisation...
🧹 2/3 Suppression des doublons...
   → 2 doublons retirés.
 3/3 Tokenisation + suppression des stopwords...


100%|██████████| 7498/7498 [00:01<00:00, 5166.12it/s]

✅ Nettoyage terminé : 7491 reviews prêtes.
